### GFG: AdaBoost vs Gradient Boosting

| Features | Adaptive Boost | Graident Boosting
|--|--|--|
| Weight Update Strategy | Increases weights of misclassified samples so that the next learner focuses more on them. | Updates predictions by minimizing a loss function using the negative gradient. |
| Base Learners | AdaBoost uses simple decision trees with one split known as the decision stumps of weak learners. | Gradient Boosting can use a wide range of base learners such as decision trees and linear models. |
| Sensitive of Noise | AdaBoost is more sensitive to noisy data and outliers due to aggressive weighting. | Gradient Boosting is less sensitive as it smooths updates using gradients. |
| Boosting Mechanism | No explicit loss function i.e it focuses on classification error. | Learners are trained sequentially with residual fitting (gradient descent). |
| Interpretability | Easier to interpret due to simple weak learners. | Harder to interpret if complex models are used. |
| Use case | Suitable for clean datasets with fewer outliers. | Suitable for complex problems with varying loss function. |

In [13]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

%matplotlib inline

### Preparing the dataset like earlier

In [14]:
# Load the data
train = pd.read_csv('../data/leaf-classification/train.csv')
test = pd.read_csv('../data/leaf-classification/test.csv')
X = train.drop(columns=['species', 'id', 'margin1']).values
y = train['margin1'].values

# Train-Validation Split
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=123)

X_train.shape, X_valid.shape, y_train.shape, y_valid.shape

((792, 191), (198, 191), (792,), (198,))

### Node Class

In [15]:
class Node:
    def __init__(self, feature_idx=None, threshold=None, info_gain=None, left=None, right=None, value=None):
        # A Node can either be a Decision Node or a Pure Node
        
        # Decision Node: Splitting data based on a feature with some threshold
        self.feature_idx = feature_idx
        self.threshold = threshold
        self.info_gain = info_gain
        self.left = left
        self.right = right

        # Leaf Node: Pure, hence only has which class it belongs to
        self.value = value

### Constituent Decision Tree Regressor from Prev Notebooks

In [16]:
class RegressionTree:
    def __init__(self, min_samples_split=2, max_depth=2):
        # Min Samples for splitting to occur
        self.min_samples_split=min_samples_split
        # Max depth for base case and preventing overfitting
        self.max_depth = max_depth

    def build_tree(self, X, y, curr_depth=0):
        n_samples, n_features = X.shape

        # Must not be base condition 
        if n_samples >= self.min_samples_split and curr_depth < self.max_depth:
            best_split = self.best_split(X, y)

            # Check if valid split was found
            if best_split["feature_idx"] is not None:
                left_node = self.build_tree(best_split["left_X"], best_split["left_y"], curr_depth + 1)
                right_node = self.build_tree(best_split["right_X"], best_split["right_y"], curr_depth + 1)
    
                return Node(best_split["feature_idx"], best_split["threshold"], best_split["loss"], left_node, right_node)

        # Returning a Pure Node with AVG value
        return Node(value=np.mean(y))

    # Can be Vectorized as well
    def predict(self, X):
        predictions = [self.predict_value(row, self.root) for row in X]
        return predictions

    def predict_value(self, x, node):
        # Leaf Node: Base case
        if node.value is not None:
            return node.value
        # Decision Node
        feature_value = x[node.feature_idx] 
        if feature_value <= node.threshold:
            return self.predict_value(x, node.left)
        else:
            return self.predict_value(x, node.right)

    def fit(self, X, y):
        self.root = self.build_tree(X, y)

In [17]:
def best_split(self, X, y):
    b_split = {
        "feature_idx": None,
        "threshold": None,
        "loss": float('inf'),
        "left_X": None,
        "left_y": None,
        "right_X": None,
        "right_y": None
    }

    y_reshaped = y.reshape(-1, 1)

    for feature_idx in range(X.shape[-1]):

        # Making all masks of feature_idx over all possible thresholds
        thresholds = np.unique(X[:, feature_idx], axis=0)                   # (thresholds,)
        left_mask = X[:, feature_idx].reshape(-1, 1) <= thresholds          # (rows, 1) <= (thresholds) = (rows, thresholds)
        right_mask = ~left_mask

        # Need to find the Sum Squared Error
        left_counts = np.sum(left_mask, axis=0)                             # (rows, thresholds).sum(axis=0) = (thresholds)
        right_counts = np.sum(right_mask, axis=0)

        valid_splits = (left_counts > 0) & (right_counts >0)

        if not np.any(valid_splits):
            continue

        left_sums = np.sum(left_mask * y_reshaped, axis=0)
        right_sums = np.sum(right_mask * y_reshaped, axis=0)

        left_means = np.divide(left_sums, left_counts, out=np.zeros_like(left_sums, dtype=float), where=left_counts!=0)
        right_means = np.divide(right_sums, right_counts, out=np.zeros_like(right_sums, dtype=float), where=right_counts!=0)

        left_sse = np.sum(np.square(y_reshaped - left_means) * left_mask, axis=0)
        right_sse = np.sum(np.square(y_reshaped - right_means) * right_mask, axis=0)

        loss = left_sse + right_sse

        loss[~valid_splits] = float('inf')
        best_thresh_idx = np.argmin(loss)

        if loss[best_thresh_idx] < b_split["loss"] and loss[best_thresh_idx] != float('inf'):
            winning_mask = left_mask[:, best_thresh_idx]
            b_split["feature_idx"] = feature_idx
            b_split["threshold"] = thresholds[best_thresh_idx]
            b_split["loss"] = loss[best_thresh_idx]
            b_split["left_X"] = X[winning_mask]
            b_split["left_y"] = y[winning_mask]
            b_split["right_X"] = X[~winning_mask]
            b_split["right_y"] = y[~winning_mask]

    return b_split
RegressionTree.best_split = best_split

In [18]:
rt = RegressionTree(min_samples_split=7, max_depth=5)
s = time.time()
rt.fit(X_train, y_train)
e = time.time()

y_pred = rt.predict(X_valid)
rmse = np.sqrt(np.mean(np.square(y_valid- y_pred)))
(f"RMSE: {rmse}"), e - s

('RMSE: 0.00973388382335118', 1.1566128730773926)